# Aggregate customer data

Combines the responses from the old SharePoint survey with the responses from the Django survey into one table.

| Table | Role |
|---|---|
| `static_data.old_responses` | input, loaded once by `fetching_data/import_old_data.ipynb` |
| `public.survey_satisfactionsurveyresponse` | input, Django's survey table |
| `processed_data.aggregated_responses` | output, **replaced** on every run |

The output still has student numbers; `new_proccess_data.ipynb` replaces them with an anonymous `visitor_id`.

In [ ]:
from pis_lib.connect import get_pg_engine
import pandas as pd
import numpy as np

In [ ]:
pg = get_pg_engine()

## Load

Both tables have the same columns, except Django's `id`.

In [ ]:
new_responses_df = pd.read_sql("SELECT * FROM public.survey_satisfactionsurveyresponse", pg)
old_responses_df = pd.read_sql("SELECT * FROM static_data.old_responses", pg)

## Combine

- Django's `id` is dropped: the old responses have none.
- Django saves a missing student number, username or course as an empty string. These become NULL, as in the old
  data, so a missing value is NULL everywhere.
- `workshop` and `used_ai` become pandas' `boolean` type: true/false with `<NA>` where the question wasn't asked or answered.
  pandas' plain `bool` can't hold missing values, so it would read them as `object`.
- `created_at` stays in UTC, as stored, and is made timezone-aware explicitly: with no new responses, pandas can't
  tell the empty table's column type, and the output would otherwise get a timestamp without time zone. `new_proccess_data.ipynb` converts it to Copenhagen time where it matters.

In [ ]:
aggregated_responses_df = pd.concat([old_responses_df, new_responses_df], ignore_index=True).drop(columns=["id"])

aggregated_responses_df = (
    aggregated_responses_df
    .replace(r'^\s*$', np.nan, regex=True)
    .astype({"workshop": "boolean", "used_ai": "boolean"})
    .assign(created_at=lambda d: pd.to_datetime(d["created_at"], utc=True))
)

print(len(old_responses_df), "old +", len(new_responses_df), "new =", len(aggregated_responses_df), "responses")

## Write to `processed_data`

> **Warning:** `if_exists="replace"` drops and recreates the table on every run.

In [ ]:
aggregated_responses_df.to_sql("aggregated_responses", pg, schema="processed_data", if_exists="replace", index=False)